# 一次没找全时如何补查

有些问题需要两处资料，第一次结果相关却只能回答一半。Iterative Retrieval（迭代检索）先检查已找到的内容，再根据明确缺口组成补查问题。

## 原理与实验设置

第二次查询只能来自用户问题和首轮实际资料，不能从参考答案或预期页取词。记录首轮页面、缺少的内容、补查问题、新增证据与停止原因；没有明确缺口、没有新增内容或达到次数上限时停止。

本页最多补查一次：高斯混合题使用首轮概念，优化方法比较题与 ROC 面积题使用尚未回答的要点，决策树题检查概念过宽时是否仍有收益。补查结果与原问题直接取两条资料比较，固定最终片段数和字符上限，并打印实际字符数；代价是多一次检索。


In [1]:
import re
import sys
from pathlib import Path


def find_course_root(start: Path) -> Path:
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import build_bm25_chunk_search, load_query_catalog, load_default_chunks
from common.nontraining_utils import load_annotation

cases = {case["id"]: case for case in load_query_catalog()}
chunks = load_default_chunks()
search = build_bm25_chunk_search(chunks)


def merge_results(*groups):
    merged = []
    seen = set()
    for group in groups:
        for item in group:
            if item.chunk_id not in seen:
                seen.add(item.chunk_id)
                merged.append(item)
    return merged


def page_coverage(results, expected_pages):
    expected = set(expected_pages)
    found = {page for item in results for page in item.pages}
    return len(expected & found) / len(expected), sorted(expected & found)


def clean_text(text):
    value = re.sub(r"→_→.*?←_←", "", str(text), flags=re.S)
    return re.sub(r"[\x00-\x1f\x7f-\x9f]", "", value).strip()


def limit_context(results, char_limit):
    limited = []
    remaining = char_limit
    for item in results:
        text = clean_text(item.text)[:max(remaining, 0)]
        limited.append(type(item)(item.chunk_id, item.pages, text, item.score))
        remaining -= len(text)
    return limited


def context_chars(results):
    return sum(len(item.text) for item in results)


def show(label, results):
    print(label, [item.pages[0] for item in results])


print("已读取片段：", len(chunks))

已读取片段： 987


## 例一：根据首轮出现的概念补查

问题先要说明高斯混合聚类怎样判断样本所属的簇，再说明参数怎样继续更新。第一次只取一个片段时，资料只交代了高斯混合模型的背景，没有给出 EM 更新和停止条件。

首轮结果已经出现“期望最大”这个明确概念，因此第二次只用首轮概念和用户原问题中的“迭代、停止条件”补查，不从参考答案或预期页码取词。

In [2]:
gmm_case = cases["gmm_em_cluster_assignment"]
gmm_first_raw = search(gmm_case["query"], top_k=1)
gmm_first = limit_context(gmm_first_raw, 512)
gmm_same_budget = limit_context(search(gmm_case["query"], top_k=2), 512)
gmm_em_concept = "期望最大" if "期望最大" in gmm_first_raw[0].text else ""
gmm_gap_terms = [term for term in ("迭代", "停止条件") if term in gmm_case["query"]]
gmm_follow_up_query = " ".join([gmm_em_concept, *gmm_gap_terms])
gmm_second = search(gmm_follow_up_query, top_k=1)
gmm_combined = limit_context(merge_results(gmm_first_raw, gmm_second), 512)
gmm_context_cap = min(context_chars(gmm_same_budget), context_chars(gmm_combined))
gmm_same_budget = limit_context(gmm_same_budget, gmm_context_cap)
gmm_combined = limit_context(gmm_combined, gmm_context_cap)
gmm_annotation = load_annotation(gmm_case["id"])

def gmm_answer_coverage(results):
    text = "".join(item.text for item in results)
    cluster_basis = all(term in text for term in ("样本归到底属于哪个簇", "高斯混合模型"))
    em_update = all(term in text for term in ("计算γji", "更新", "反复迭代", "停止条件"))
    return (cluster_basis + em_update) / 2

def gmm_answer_points(results):
    text = "".join(item.text for item in results)
    return {
        "样本所属簇的判断背景": all(term in text for term in ("样本归到底属于哪个簇", "高斯混合模型")),
        "EM 参数更新并迭代到停止条件": all(term in text for term in ("计算γji", "更新", "反复迭代", "停止条件")),
    }

show("第一次：", gmm_first)
show("原问题直接取两条：", gmm_same_budget)
print("首轮出现的概念：", gmm_em_concept, "；补查问题：", gmm_follow_up_query)
show("补查后：", gmm_combined)
print("最终资料量（原问题直接取两条 / 补查合并）：", len(gmm_same_budget), "/", len(gmm_combined), "；字符上限：", gmm_context_cap, "；实际字符数：", context_chars(gmm_same_budget), "→", context_chars(gmm_combined))
print("检索次数：1 → 2")
print("原问题直接取两条的覆盖：", page_coverage(gmm_same_budget, gmm_annotation["expected_pages"]))
print("按首轮概念补查后的覆盖：", page_coverage(gmm_combined, gmm_annotation["expected_pages"]))
print("必要回答要点（直接取两条 → 补查后）：")
gmm_before_points = gmm_answer_points(gmm_same_budget)
gmm_after_points = gmm_answer_points(gmm_combined)
for name in gmm_before_points:
    print(f"  {name}：{gmm_before_points[name]} → {gmm_after_points[name]}")
print("回答要点覆盖率：", gmm_answer_coverage(gmm_same_budget), "→", gmm_answer_coverage(gmm_combined))
assert len(gmm_same_budget) == len(gmm_combined) == 2 and context_chars(gmm_same_budget) == context_chars(gmm_combined) == gmm_context_cap
assert gmm_em_concept and gmm_gap_terms == ["迭代", "停止条件"] and gmm_answer_coverage(gmm_same_budget) == 0.5 and gmm_answer_coverage(gmm_combined) == 1.0

第一次： [113]
原问题直接取两条： [113, 51]
首轮出现的概念： 期望最大 ；补查问题： 期望最大 迭代 停止条件
补查后： [113, 118]
最终资料量（原问题直接取两条 / 补查合并）： 2 / 2 ；字符上限： 512 ；实际字符数： 512 → 512
检索次数：1 → 2
原问题直接取两条的覆盖： (0.5, [113])
按首轮概念补查后的覆盖： (1.0, [113, 118])
必要回答要点（直接取两条 → 补查后）：
  样本所属簇的判断背景：True → True
  EM 参数更新并迭代到停止条件：False → True
回答要点覆盖率： 0.5 → 1.0


结果从只找到高斯混合模型的背景，变为同时找到样本归类依据和 EM 更新、停止条件；必要页面覆盖由 50% 提高到 100%，最终都保留 2 个片段并使用相同字符上限，资料量可比较。第二次检索不是重复原问题，而是使用首轮实际出现的“期望最大”和用户问题中的缺口去找尚缺的结论。

## 例二：根据回答缺口补查

回答优化方法比较题需要两部分内容：第 38 页的梯度下降更新式，以及第 39 页的牛顿法更新式与拟牛顿法说明。原问题直接取两个片段时，第一条已经找到牛顿法与拟牛顿法，第二条却只有两种方法的概念比较，仍缺少梯度下降更新式。

第二次检索从原问题中取出第一轮还没有回答完整的部分：“梯度下降法、迭代公式、步长、学习率”。两种做法最终都保留两个片段，但补查会多发起一次检索。

In [3]:
newton_case = cases["newton_methods_comparison"]
newton_first_raw = search(newton_case["query"], top_k=1)
newton_first = limit_context(newton_first_raw, 512)
newton_same_budget = limit_context(search(newton_case["query"], top_k=2), 512)
gradient_formula_terms = ("梯度下降法", "迭代公式", "步长", "学习率")
first_text = newton_first_raw[0].text
gradient_formula_missing = not all(term in first_text for term in gradient_formula_terms)
newton_follow_up_query = " ".join(
    term for term in gradient_formula_terms if term in newton_case["query"]
) if gradient_formula_missing else ""
newton_second = search(newton_follow_up_query, top_k=1) if newton_follow_up_query else []
newton_combined = limit_context(merge_results(newton_first_raw, newton_second), 512)
newton_context_cap = min(context_chars(newton_same_budget), context_chars(newton_combined))
newton_same_budget = limit_context(newton_same_budget, newton_context_cap)
newton_combined = limit_context(newton_combined, newton_context_cap)
newton_annotation = load_annotation(newton_case["id"])


def answer_part_coverage(results):
    text = "".join(item.text for item in results)
    gradient_part = all(word in text for word in ("xt+1=xt−a∇f", "步长", "学习率"))
    newton_part = all(word in text for word in ("牛顿法的迭代公式", "Hessian矩阵的逆矩阵", "近似逆矩阵", "拟牛顿法"))
    return (gradient_part + newton_part) / 2

def newton_answer_parts(results):
    text = "".join(item.text for item in results)
    return {
        "梯度下降更新式、步长和学习率": all(word in text for word in ("xt+1=xt−a∇f", "步长", "学习率")),
        "牛顿法代价与拟牛顿法": all(word in text for word in ("牛顿法的迭代公式", "Hessian矩阵的逆矩阵", "近似逆矩阵", "拟牛顿法")),
    }

show("第一次：", newton_first)
show("原问题直接取两条：", newton_same_budget)
print("补查问题：", newton_follow_up_query)
show("补查后：", newton_combined)
newton_direct_pages = [item.pages[0] for item in newton_same_budget]
newton_follow_up_pages = [item.pages[0] for item in newton_combined]
if newton_direct_pages == newton_follow_up_pages:
    print("两边页码相同，但补查替换了片段：")
    for position, (before_item, after_item) in enumerate(zip(newton_same_budget, newton_combined), start=1):
        if before_item.chunk_id != after_item.chunk_id:
            print(f"  第 {position} 条：{before_item.chunk_id} → {after_item.chunk_id}（页 {before_item.pages[0]}）")
print("最终资料量（原问题直接取两条 / 补查合并）：", len(newton_same_budget), "/", len(newton_combined))
newton_before_parts = newton_answer_parts(newton_same_budget)
newton_after_parts = newton_answer_parts(newton_combined)
print("必要回答要点（直接取两条 → 补查后）：")
for name in newton_before_parts:
    print(f"  {name}：{newton_before_parts[name]} → {newton_after_parts[name]}")
print("原问题直接取两条的回答要点覆盖率：", answer_part_coverage(newton_same_budget))
print("按回答缺口补查后的覆盖率：", answer_part_coverage(newton_combined))
print("检索次数：1 →", 1 + bool(newton_follow_up_query), "；字符上限：", newton_context_cap, "；实际字符数：", context_chars(newton_same_budget), "→", context_chars(newton_combined))
assert gradient_formula_missing and all(term in newton_case["query"] for term in newton_follow_up_query.split())
assert len(newton_same_budget) == len(newton_combined) == 2 and context_chars(newton_same_budget) == context_chars(newton_combined) == newton_context_cap and answer_part_coverage(newton_same_budget) == 0.5 and answer_part_coverage(newton_combined) == 1.0
assert newton_direct_pages == newton_follow_up_pages == [39, 38] and newton_same_budget[1].chunk_id != newton_combined[1].chunk_id

第一次： [39]
原问题直接取两条： [39, 38]
补查问题： 梯度下降法 迭代公式 步长 学习率
补查后： [39, 38]
两边页码相同，但补查替换了片段：
  第 2 条：c249 → c248（页 38）
最终资料量（原问题直接取两条 / 补查合并）： 2 / 2
必要回答要点（直接取两条 → 补查后）：
  梯度下降更新式、步长和学习率：False → True
  牛顿法代价与拟牛顿法：True → True
原问题直接取两条的回答要点覆盖率： 0.5
按回答缺口补查后的覆盖率： 1.0
检索次数：1 → 2 ；字符上限： 510 ；实际字符数： 510 → 510


补查后，两个片段分别包含梯度下降更新式，以及牛顿法和拟牛顿法的说明。必要内容从只找到一部分变为全部找到，最终片段数和字符上限与直接检索相同；下面用另外两题检查适用范围。


## 第二次检查：ROC 面积和决策树停止条件

再用两个不同的问题做短检查。ROC 题按回答缺口补查后找回了面积计算；决策树题中的“递归返回”过于宽泛，补查没有增加回答内容，用它说明这种做法什么时候不值得继续。

In [4]:
# ROC 题：第一条只有坐标增量，面积计算还没有出现。
roc_case = cases["roc_threshold_process"]
roc_first = search(roc_case["query"], top_k=1)
roc_same_budget = search(roc_case["query"], top_k=2)
roc_first_text = roc_first[0].text
# 每个候选词只检查用户问题和 roc_first 的首轮文字，不读取 top2 对照的第二条。
roc_candidate_terms = ("预测值", "线段", "面积")
roc_term_sources = {}
for term in roc_candidate_terms:
    sources = []
    if term in roc_case["query"]:
        sources.append("用户问题")
    if term in roc_first_text:
        sources.append("首轮结果")
    roc_term_sources[term] = sources
roc_follow_up_terms = [term for term in roc_candidate_terms if roc_term_sources[term]]
roc_follow_up_query = " ".join(roc_follow_up_terms)
roc_second = search(roc_follow_up_query, top_k=1)
roc_combined = limit_context(merge_results(roc_first, roc_second), 512)
roc_same_budget = limit_context(roc_same_budget, 512)
roc_context_cap = min(context_chars(roc_same_budget), context_chars(roc_combined))
roc_same_budget = limit_context(roc_same_budget, roc_context_cap)
roc_combined = limit_context(roc_combined, roc_context_cap)
roc_annotation = load_annotation(roc_case["id"])

def roc_answer_parts(results):
    text = "".join(item.text for item in results)
    return {
        "假正例和真正例改变坐标": all(term in text for term in ("假正例", "真正例", "步长")),
        "线段面积计算": all(term in text for term in ("线段", "面积")) and any(term in text for term in ("矩形", "梯形")),
    }

roc_before_parts = roc_answer_parts(roc_same_budget)
roc_after_parts = roc_answer_parts(roc_combined)
roc_before_chars = sum(len(item.text) for item in roc_same_budget)
roc_after_chars = sum(len(item.text) for item in roc_combined)
show("ROC 第一次：", roc_first)
show("ROC 原问题直接取两条：", roc_same_budget)
print("ROC 补查词来源：")
for term in roc_candidate_terms:
    source = "、".join(roc_term_sources[term]) or "未加入（两处都没有）"
    print(f"  {term}：{source}")
print("ROC 补查问题（只用有来源的词）：", roc_follow_up_query)
show("ROC 补查后：", roc_combined)
print("必要页面覆盖（直接取两条 → 补查后）：", page_coverage(roc_same_budget, roc_annotation["expected_pages"]), "→", page_coverage(roc_combined, roc_annotation["expected_pages"]))
print("必要回答要点（直接取两条 → 补查后）：")
for name in roc_before_parts:
    print(f"  {name}：{roc_before_parts[name]} → {roc_after_parts[name]}")
print("检索次数：1 → 2；最终资料量：", len(roc_same_budget), "/", len(roc_combined))
print("字符上限：", roc_context_cap, "；上下文实际字符数：", roc_before_chars, "→", roc_after_chars)
assert roc_follow_up_terms == ["线段", "面积"] and "预测值" not in roc_follow_up_query and "AUC" not in roc_follow_up_query and "梯形公式" not in roc_follow_up_query
assert [item.pages[0] for item in roc_same_budget] == [21, 21] and [item.pages[0] for item in roc_combined] == [21, 22]
assert len(roc_same_budget) == len(roc_combined) == 2 and roc_before_chars == roc_after_chars == roc_context_cap and not roc_before_parts["线段面积计算"] and roc_after_parts["线段面积计算"]

# 决策树题说明限制：首轮概念太宽泛时，再查一次也未必补得更完整。
tree_case = cases["decision_tree_stop_conditions"]
tree_result_k = 3
tree_first = search(tree_case["query"], top_k=1)
tree_direct = search(tree_case["query"], top_k=tree_result_k)
tree_concept = "递归返回" if "递归返回" in tree_first[0].text else ""
tree_gap_terms = [term for term in ("每种情况", "分支", "类别") if term in tree_case["query"]]
tree_follow_up_query = " ".join([tree_concept, *tree_gap_terms])
tree_follow_up = search(tree_follow_up_query, top_k=tree_result_k)
tree_combined = merge_results(tree_first, tree_follow_up)[:tree_result_k]
tree_context_cap = min(context_chars(tree_direct), context_chars(tree_combined))
tree_direct = limit_context(tree_direct, tree_context_cap)
tree_combined = limit_context(tree_combined, tree_context_cap)
tree_annotation = load_annotation(tree_case["id"])

def tree_answer_parts(results):
    text = "".join(item.text for item in results)
    return {
        "子集只含一类时停止": all(term in text for term in ("某一类", "无需再进行递归划分")),
        "属性用完时按后验分布确定类别": all(term in text for term in ("属性集合A", "后验分布")),
        "空分支按样本最多的类别处理": all(term in text for term in ("分支", "样本最多的类", "先验分布")),
    }

tree_before_parts = tree_answer_parts(tree_direct)
tree_after_parts = tree_answer_parts(tree_combined)
show("决策树原问题直接取三条：", tree_direct)
print("首轮出现的概念：", tree_concept, "；补查问题：", tree_follow_up_query)
show("决策树补查后：", tree_combined)
print("必要回答要点（直接取三条 → 补查后）：")
for name in tree_before_parts:
    print(f"  {name}：{tree_before_parts[name]} → {tree_after_parts[name]}")
print("检索次数：1 → 2；最终资料量：", len(tree_direct), "/", len(tree_combined), "；实际字符数：", context_chars(tree_direct), "→", context_chars(tree_combined))
print("限制结论：补查没有增加回答要点，保留原结果并停止。")
assert tree_concept and tree_gap_terms == ["每种情况", "分支", "类别"] and len(tree_direct) == len(tree_combined) == tree_result_k and context_chars(tree_direct) == context_chars(tree_combined) == tree_context_cap and tree_before_parts == tree_after_parts

ROC 第一次： [21]
ROC 原问题直接取两条： [21, 21]
ROC 补查词来源：
  预测值：未加入（两处都没有）
  线段：用户问题
  面积：用户问题
ROC 补查问题（只用有来源的词）： 线段 面积
ROC 补查后： [21, 22]
必要页面覆盖（直接取两条 → 补查后）： (0.5, [21]) → (1.0, [21, 22])
必要回答要点（直接取两条 → 补查后）：
  假正例和真正例改变坐标：True → True
  线段面积计算：False → True
检索次数：1 → 2；最终资料量： 2 / 2
字符上限： 418 ；上下文实际字符数： 418 → 418
决策树原问题直接取三条： [45, 45, 45]
首轮出现的概念： 递归返回 ；补查问题： 递归返回 每种情况 分支 类别
决策树补查后： [45, 45, 51]
必要回答要点（直接取三条 → 补查后）：
  子集只含一类时停止：True → True
  属性用完时按后验分布确定类别：False → False
  空分支按样本最多的类别处理：False → False
检索次数：1 → 2；最终资料量： 3 / 3 ；实际字符数： 749 → 749
限制结论：补查没有增加回答要点，保留原结果并停止。


ROC 题补查后找回了线段面积计算。决策树题则没有改善：首轮出现的“递归返回”太宽泛，用它和用户原话继续查询并没有补全另外两种停止情况，因此保留原结果并停止。这个反例说明，只有首轮概念能明确指向缺口时才值得补查。

## 结果边界与扩展接口

高斯混合、牛顿法比较与 ROC 面积题补回了明确缺口；决策树首轮的“递归返回”过于宽泛，再查一次仍没找全。因此先判断缺口能否组成有方向的查询，再决定补查。首轮方向错误时回到[改写检索问题](../4.%20检索阶段/改写检索问题.ipynb)。

代码去重，并使前后结果遵守同一片段数与字符预算；页面覆盖仅在检索后核对。下面的参考函数将“检索 → 检查缺口 → 补查 → 合并后停止”抽成接口，不调用外部模型。接入生成器时，草答可用于发现缺口，新事实仍需回到原文核实。


In [5]:
from common.eval_utils import emit_tutorial_audit

# 统一保存契约：页码来自首轮/补查的真实片段，标注只在检索完成后读取。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        values = item.pages if hasattr(item, 'pages') else [item.page]
        for page in values:
            page = int(page)
            if page not in pages:
                pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _emit(method, role, case_id, before_items, after_items, purpose=None):
    annotation = load_annotation(case_id)
    payload = {'case_id': case_id, 'method': method, 'role': role,
              'before': _metrics(before_items, annotation['expected_pages']),
              'after': _metrics(after_items, annotation['expected_pages'])}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

# 迭代方法的 before 是首轮真实返回，after 是首轮与补查合并后的真实返回。
_emit('按回答缺口补查', 'main', 'newton_methods_comparison', newton_first, newton_combined)
_emit('按回答缺口补查', 'check', 'roc_threshold_process', roc_first, roc_combined, '再次改善')
_emit('按首轮概念补查', 'main', 'gmm_em_cluster_assignment', gmm_first, gmm_combined)
_emit('按首轮概念补查', 'check', 'decision_tree_stop_conditions', tree_first, tree_combined, '说明不适用或限制')


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[检查是否应继续检索](检查检索结果后再继续.ipynb)

